# 🛠️ Step 2: Specialized Agents with Tools

## What we're building

Instead of one agent doing everything badly, we'll create **specialized agents** that each:
- Have a focused responsibility
- Own specific tools (infrastructure APIs)
- Produce structured outputs for the next agent

## Our Agents

| Agent | Role | Tools |
|-------|------|-------|
| **Triage Agent** | Classify severity, check history, get runbook | `check_alert_history`, `get_runbook` |
| **Diagnostics Agent** | Find root cause using metrics and logs | `get_metrics`, `get_logs`, `check_dependencies` |
| **Remediation Agent** | Execute the fix | `restart_pod`, `scale_service`, `flush_cache`, `toggle_feature_flag` |
| **Verification Agent** | Confirm the fix worked | `get_health_status`, `run_smoke_test` |
| **Comms Agent** | Notify team, create tickets | `post_to_slack`, `create_incident_ticket`, `update_status_page` |

---

## 💡 Key Concept: Task Decomposition

Each agent is an **expert** at one thing. This means:
- Simpler instructions (less confusion)
- Focused tool sets (less chance of misuse)
- Testable in isolation
- Replaceable/upgradable independently

In [ ]:
import os
import json
from dotenv import load_dotenv
from agent_framework import ChatAgent
from agent_framework.azure import AzureAIAgentClient
from azure.identity.aio import AzureCliCredential

# Import our mock infrastructure tools
import sys
sys.path.insert(0, "..")

from tools.mock_infra import (
    # Triage tools
    check_alert_history,
    get_runbook,
    # Diagnostics tools
    get_metrics,
    get_logs,
    check_dependencies,
    # Remediation tools
    restart_pod,
    scale_service,
    flush_cache,
    toggle_feature_flag,
    # Verification tools
    get_health_status,
    run_smoke_test,
    # Communications tools
    post_to_slack,
    create_incident_ticket,
    update_status_page,
load_dotenv("../.env")
)

with open("../data/incidents.json") as f:

# Load incident data
with open("data/incidents.json") as f:
    incidents = json.load(f)


alert = incidents[0]  # payment-api high latencyprint(f"   Service: {alert['service']} | Severity: {alert['severity']}")
print(f"🔴 Working with alert: {alert['title']}")

## Agent 1: Triage Agent

The Triage Agent is the first responder. Its job:
1. Check if this is a known/recurring issue
2. Look up the relevant runbook
3. Classify whether automated remediation is possible

### 🎯 YOUR TASK
Fill in the `instructions` for the Triage Agent. It should:
- Use `check_alert_history` to see if this happened before
- Use `get_runbook` to find the remediation playbook
- Output a clear triage summary with: severity, whether it's recurring, and recommended action

In [ ]:
async def build_triage_agent():
    async with (
        AzureCliCredential() as credential,
        AzureAIAgentClient(async_credential=credential) as client,
    ):
        # TODO: Write instructions for the Triage Agent
        # Hint: Tell it its role, what tools it has, and what output format you expect
        triage_instructions = """
You are an incident Triage Agent. Your job is to be the first responder when an alert fires.

When given an alert:
1. Use check_alert_history to see if this service has had similar issues recently
2. Use get_runbook to look up the standard operating procedure for this type of incident
3. Produce a triage summary with:
   - Severity assessment (critical/high/medium/low)
   - Whether this is a recurring issue
   - Whether automated remediation is allowed (from the runbook)
   - Recommended next steps for the Diagnostics Agent

Be concise and factual. Do not guess at root causes — that's the Diagnostics Agent's job.
"""

        triage_agent = client.create_agent(
            name="TriageAgent",
            instructions=triage_instructions,
            tools=[check_alert_history, get_runbook],
        )
        
        # Run the triage agent on our alert
        triage_prompt = f"""New alert received:
Title: {alert['title']}
Service: {alert['service']}
Severity: {alert['severity']}
Description: {alert['description']}

Triage this incident."""

        result = await triage_agent.run(triage_prompt)
        print("\n📋 TRIAGE RESULT:\n")
        print(result.text)
        return result.text

triage_output = await build_triage_agent()

## Agent 2: Diagnostics Agent

The Diagnostics Agent digs into the technical details. Its job:
1. Pull real metrics (CPU, memory, latency, error rates)
2. Analyze logs for error patterns
3. Check dependency health
4. Identify the root cause

### 🎯 YOUR TASK
Fill in the `instructions` for the Diagnostics Agent. It should:
- Use its tools to gather evidence (not guess!)
- Correlate findings across metrics, logs, and dependencies
- Output a root cause analysis with specific evidence

In [ ]:
async def build_diagnostics_agent(triage_context: str):
    async with (
        AzureCliCredential() as credential,
        AzureAIAgentClient(async_credential=credential) as client,
    ):
        # TODO: Write instructions for the Diagnostics Agent
        # Hint: It should systematically gather evidence using all 3 tools
        diagnostics_instructions = """
You are an incident Diagnostics Agent. Your job is to identify the root cause of an incident.

You have access to real monitoring data. When investigating:
1. Use get_metrics to check CPU, memory, latency, and error rates for the affected service
2. Use get_logs to find error messages and patterns
3. Use check_dependencies to see if upstream/downstream services are healthy

Your output should be a root cause analysis with:
- What is failing (specific pod, specific component)
- Why it's failing (evidence from metrics/logs)
- What's the blast radius (which other services are affected)
- Recommended remediation action with specific parameters
  (e.g., "restart payment-api-pod-3" not just "restart a pod")

IMPORTANT: Base your analysis ONLY on data from your tools. Do not speculate.
"""

        diagnostics_agent = client.create_agent(
            name="DiagnosticsAgent",
            instructions=diagnostics_instructions,
            tools=[get_metrics, get_logs, check_dependencies],
        )

        # Pass triage context + original alert
        diag_prompt = f"""Investigate this incident. The Triage Agent has already assessed it:

--- TRIAGE SUMMARY ---
{triage_context}

--- ORIGINAL ALERT ---
Service: {alert['service']}
Description: {alert['description']}

Run diagnostics and identify the root cause."""

        result = await diagnostics_agent.run(diag_prompt)
        print("\n🔍 DIAGNOSTICS RESULT:\n")
        print(result.text)
        return result.text

diagnostics_output = await build_diagnostics_agent(triage_output)

## Agent 3: Remediation Agent

The Remediation Agent takes action. It has the "dangerous" tools that actually change infrastructure.

### 🎯 YOUR TASK
Fill in the `instructions` for the Remediation Agent. Key considerations:
- It should ONLY act based on the diagnostics findings (not guess)
- It should check the runbook's `auto_remediation_allowed` flag
- If auto-remediation isn't allowed, it should call `escalate_to_human` instead

In [ ]:
async def build_remediation_agent(diagnostics_context: str):
    async with (
        AzureCliCredential() as credential,
        AzureAIAgentClient(async_credential=credential) as client,
    ):
        # TODO: Write instructions for the Remediation Agent
        remediation_instructions = """
You are an incident Remediation Agent. Your job is to execute fixes based on the diagnosis.

You have access to infrastructure tools:
- restart_pod: Restart a specific pod (use when OOM/memory leak detected)
- scale_service: Scale to more replicas (use when CPU is high or traffic spike)
- flush_cache: Flush a cache (use when stale data suspected)
- toggle_feature_flag: Enable/disable features (use for graceful degradation)
- escalate_to_human: Page a human when auto-fix isn't possible

Rules:
1. ONLY take actions that the diagnostics directly support
2. If the runbook says auto_remediation_allowed=False, call escalate_to_human
3. Be specific: use exact pod names, exact service names, exact flag names from the diagnostics
4. Report what actions you took and their results

Do NOT take multiple unrelated actions. Fix the root cause identified by diagnostics.
"""

        remediation_agent = client.create_agent(
            name="RemediationAgent",
            instructions=remediation_instructions,
            tools=[restart_pod, scale_service, flush_cache, toggle_feature_flag, escalate_to_human],
        )

        remediation_prompt = f"""Execute remediation based on this diagnosis:

--- DIAGNOSTICS FINDINGS ---
{diagnostics_context}

--- SERVICE ---
{alert['service']}

Take the appropriate remediation action now."""

        result = await remediation_agent.run(remediation_prompt)
        print("\n🔧 REMEDIATION RESULT:\n")
        print(result.text)
        return result.text

remediation_output = await build_remediation_agent(diagnostics_output)

## Agent 4: Verification Agent

After remediation, we need to verify the fix actually worked.

### 🎯 YOUR TASK
Create the Verification Agent. It should:
- Check service health after the fix
- Run smoke tests
- Report whether the incident is resolved or needs escalation

In [ ]:
async def build_verification_agent(remediation_context: str):
    async with (
        AzureCliCredential() as credential,
        AzureAIAgentClient(async_credential=credential) as client,
    ):
        # TODO: Write the instructions and create the agent
        verification_instructions = """
You are a Verification Agent. After remediation actions are taken, you verify the fix worked.

Steps:
1. Use get_health_status to check if the service is now healthy
2. Use run_smoke_test to run functional tests against the service
3. Report your findings:
   - Is the service healthy? (latency, error rate, resource usage)
   - Are smoke tests passing?
   - Final verdict: RESOLVED or NEEDS_ESCALATION

If all checks pass, mark as RESOLVED.
If any checks fail, mark as NEEDS_ESCALATION with details.
"""

        verification_agent = client.create_agent(
            name="VerificationAgent",
            instructions=verification_instructions,
            tools=[get_health_status, run_smoke_test],
        )

        verify_prompt = f"""The following remediation was performed:

--- REMEDIATION ACTIONS ---
{remediation_context}

--- SERVICE ---
{alert['service']}

Verify that the service is now healthy and the incident is resolved."""

        result = await verification_agent.run(verify_prompt)
        print("\n✅ VERIFICATION RESULT:\n")
        print(result.text)
        return result.text

verification_output = await build_verification_agent(remediation_output)

## Agent 5: Communications Agent

The final agent handles all human communication.

### 🎯 YOUR TASK
Create the Comms Agent. It should:
- Post a resolution summary to Slack
- Create a post-incident ticket
- Update the status page

In [ ]:
async def build_comms_agent(triage_ctx: str, diagnostics_ctx: str, remediation_ctx: str, verification_ctx: str):
    async with (
        AzureCliCredential() as credential,
        AzureAIAgentClient(async_credential=credential) as client,
    ):
        # TODO: Write instructions for the Communications Agent
        comms_instructions = """
You are the Communications Agent. After an incident is resolved, you handle all notifications.

Your responsibilities:
1. Post a concise incident summary to Slack (#incidents channel)
2. Create a post-incident ticket with full details for the post-mortem
3. Update the status page to 'operational' if resolved

Your Slack message should be brief and include:
- What happened (1 sentence)
- Root cause (1 sentence)
- Fix applied (1 sentence)
- Current status (resolved/monitoring)

The incident ticket should have the full timeline and details.
"""

        comms_agent = client.create_agent(
            name="CommunicationsAgent",
            instructions=comms_instructions,
            tools=[post_to_slack, create_incident_ticket, update_status_page],
        )

        comms_prompt = f"""The incident has been resolved. Notify the team.

--- FULL INCIDENT TIMELINE ---

TRIAGE:
{triage_ctx}

DIAGNOSTICS:
{diagnostics_ctx}

REMEDIATION:
{remediation_ctx}

VERIFICATION:
{verification_ctx}

--- SERVICE ---
{alert['service']}

Post to Slack, create an incident ticket, and update the status page."""

        result = await comms_agent.run(comms_prompt)
        print("\n📢 COMMUNICATIONS RESULT:\n")
        print(result.text)
        return result.text

comms_output = await build_comms_agent(triage_output, diagnostics_output, remediation_output, verification_output)

## 🎉 What We've Built

We now have 5 specialized agents, each with focused tools:

```
Alert ──► Triage ──► Diagnostics ──► Remediation ──► Verification ──► Comms
          (2 tools)   (3 tools)       (5 tools)       (2 tools)       (3 tools)
```

**But there's a problem...**

We're manually passing outputs between agents. What if:
- The verification fails? We need to loop back to remediation.
- We want to run this without human intervention?
- We want to handle different incident types with different routing?

That's where **workflow orchestration** comes in.

---

## ➡️ Next: Step 3 — Workflow Orchestration

In the next notebook, we'll wire these agents into an automated `WorkflowBuilder` that:
- Chains agents together automatically
- Routes based on conditions (fix worked → comms, fix failed → retry/escalate)
- Runs the entire pipeline with one call